In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
PJME_data = pd.read_csv(
    "PJME_phase2_preprocessed.csv",
    parse_dates=["Datetime"]
)

In [3]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [4]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


print(train.index.min(), "to", train.index.max())
print(validation.index.min(), "to", validation.index.max())
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)
2002-01-08 01:00:00 to 2013-08-13 16:00:00
2013-08-13 17:00:00 to 2016-02-07 08:00:00
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [5]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [6]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = target_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = target_scaler.transform(
    y_test.values.reshape(-1, 1)
)

print("Scaling completed.")

Scaling completed.


In [7]:
def create_sequences(X, y, input_steps=168, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(
            X[i-input_steps:i]
        )

        y_seq.append(
            y[i:i+output_steps].flatten()
        )

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=168,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=168,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=168,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101465, 168, 15)
y_train: (101465, 24)
X_val: (21593, 168, 15)
y_val: (21593, 24)
X_test: (21593, 168, 15)
y_test: (21593, 24)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [9]:
gru_model_es = Sequential([
    GRU(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

gru_model_es.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_model_es.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [11]:
gru_history_es = gru_model_es.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 31s 18ms/step - loss: 0.1218 - mae: 0.2476 - val_loss: 0.0927 - val_mae: 0.2248
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0725 - mae: 0.1933 - val_loss: 0.0810 - val_mae: 0.2040
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0670 - mae: 0.1842 - val_loss: 0.0788 - val_mae: 0.2010
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0633 - mae: 0.1777 - val_loss: 0.0773 - val_mae: 0.2003
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0604 - mae: 0.1731 - val_loss: 0.0775 - val_mae: 0.2010
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0583 - mae: 0.1699 - val_loss: 0.0749 - val_mae: 0.1941
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0565 - mae: 0.1675 - val_loss: 0.0763 - val_mae: 0.1942
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0549 - mae: 0.1653 - val_loss: 0.0765 - val_mae: 0.1957
Epoch 9/15
1586/1586 ━━━

In [12]:
gru_model_es_pred_scaled = gru_model_es.predict(
    X_test_seq,
    batch_size=64
)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [13]:
gru_pred = target_scaler.inverse_transform(
    gru_model_es_pred_scaled.reshape(-1, 1)
).reshape(gru_model_es_pred_scaled.shape)

gru_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [14]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
gru_mae = mean_absolute_error(
    gru_actual.flatten(),
    gru_pred.flatten()
)

gru_rmse = np.sqrt(
    mean_squared_error(
        gru_actual.flatten(),
        gru_pred.flatten()
    )
)

gru_mape = np.mean(
    np.abs(
        (gru_actual.flatten() - gru_pred.flatten())
        / gru_actual.flatten()
    )
) * 100

gru_r2 = r2_score(
    gru_actual.flatten(),
    gru_pred.flatten()
)

gru_bias = np.mean(
    gru_pred.flatten() - gru_actual.flatten()
)

print("GRU + Earlystopping")
print("MAE :", gru_mae)
print("RMSE:", gru_rmse)
print("MAPE:", gru_mape)
print("R²  :", gru_r2)
print("Bias:", gru_bias)

GRU + Earlystopping
MAE : 1409.7967050825873
RMSE: 1980.18629570404
MAPE: 4.480879707174862
R²  : 0.9054711307453631
Bias: 244.07954247340598


In [15]:
print("Epochs completed:", len(gru_history_es.history["loss"]))

Epochs completed: 9


**Dropout**

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

gru_dropout_model = Sequential([
    GRU(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(64, activation="relu"),

    Dropout(0.2),

    Dense(24)
])

gru_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_dropout_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
gru_history_dropout = gru_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - loss: 0.1923 - mae: 0.3289 - val_loss: 0.0963 - val_mae: 0.2312
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1216 - mae: 0.2633 - val_loss: 0.0873 - val_mae: 0.2205
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1139 - mae: 0.2541 - val_loss: 0.0826 - val_mae: 0.2092
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1103 - mae: 0.2495 - val_loss: 0.0837 - val_mae: 0.2139
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1080 - mae: 0.2471 - val_loss: 0.0804 - val_mae: 0.2066
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1057 - mae: 0.2442 - val_loss: 0.0807 - val_mae: 0.2060
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1035 - mae: 0.2419 - val_loss: 0.0777 - val_mae: 0.2031
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1023 - mae: 0.2405 - val_loss: 0.0796 - val_mae: 0.2040
Epoch 9/15
1586/1586 ━━━

In [18]:
dropout_pred_scaled = gru_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [19]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [20]:
mae = mean_absolute_error(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        dropout_actual.flatten(),
        dropout_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten()
    )
) * 100

r2 = r2_score(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

bias = np.mean(
    dropout_pred.flatten() -
    dropout_actual.flatten()
)

print("== GRU + DROPOUT ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== GRU + DROPOUT ==
MAE : 1483.847755260765
RMSE: 2052.339182603539
MAPE: 4.750156463961595
R²  : 0.89845684919783
Bias: 293.26455467293727


**Batch Normalization**

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  BatchNormalization

gru_bn_model = Sequential([
    GRU(
        64,
        input_shape=(168, 15)
    ),

    BatchNormalization(),

    Dense(64, activation="relu"),

    BatchNormalization(),

    Dense(24)
])

gru_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_bn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,784 (85.09 KB)

 Trainable params: 21,528 (84.09 KB)

 Non-trainable params: 256 (1.00 KB)

In [22]:
history_bn = gru_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - loss: 0.1684 - mae: 0.3010 - val_loss: 0.1012 - val_mae: 0.2353
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0980 - mae: 0.2350 - val_loss: 0.0847 - val_mae: 0.2107
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.0902 - mae: 0.2246 - val_loss: 0.0813 - val_mae: 0.2054
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0847 - mae: 0.2171 - val_loss: 0.0817 - val_mae: 0.2096
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - loss: 0.0813 - mae: 0.2126 - val_loss: 0.0751 - val_mae: 0.1968
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0782 - mae: 0.2085 - val_loss: 0.0777 - val_mae: 0.1982
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0767 - mae: 0.2064 - val_loss: 0.0767 - val_mae: 0.1996
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 42s 14ms/step - loss: 0.0739 - mae: 0.2027 - val_loss: 0.0788 - val_mae: 0.1964
Epoch 9/15
1586/1586 ━━━

In [23]:
bn_pred_scaled = gru_bn_model.predict(
    X_test_seq,
    batch_size=64
)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [24]:
rnn_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(bn_pred_scaled.shape)

rnn_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(
    rnn_bn_actual.flatten(),
    rnn_bn_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        rnn_bn_actual.flatten(),
        rnn_bn_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (rnn_bn_actual.flatten() - rnn_bn_pred.flatten())
        / rnn_bn_actual.flatten()
    )
) * 100

r2 = r2_score(
    rnn_bn_actual.flatten(),
    rnn_bn_pred.flatten()
)

bias = np.mean(
    rnn_bn_pred.flatten() - rnn_bn_actual.flatten()
)

print("== GRU + Batch ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== GRU + Batch ==
MAE : 1491.3439278771682
RMSE: 2121.1136738033306
MAPE: 4.7553097408570695
R²  : 0.8915373404934451
Bias: 420.75683135084165


**RMSPROP**

In [26]:
from tensorflow.keras.optimizers import RMSprop

gru_rmsprop = Sequential([
    GRU(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

gru_rmsprop.compile(
    optimizer=RMSprop(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

gru_rmsprop.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_3 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history_rmsprop = gru_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1210 - mae: 0.2517 - val_loss: 0.1095 - val_mae: 0.2394
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0766 - mae: 0.1992 - val_loss: 0.0904 - val_mae: 0.2152
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0699 - mae: 0.1882 - val_loss: 0.0844 - val_mae: 0.2061
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0661 - mae: 0.1819 - val_loss: 0.0820 - val_mae: 0.2053
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0635 - mae: 0.1776 - val_loss: 0.0772 - val_mae: 0.1976
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 11ms/step - loss: 0.0615 - mae: 0.1746 - val_loss: 0.0754 - val_mae: 0.1959
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0595 - mae: 0.1715 - val_loss: 0.0782 - val_mae: 0.2040
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0581 - mae: 0.1694 - val_loss: 0.0759 - val_mae: 0.1954
Epoch 9/15
1586/1586 ━━━

In [28]:
rmsprop_pred_scaled = gru_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Prediction shape: (21593, 24)


In [29]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rmsprop_mae = mean_absolute_error(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_rmse = np.sqrt(
    mean_squared_error(
        rmsprop_actual.flatten(),
        rmsprop_pred.flatten()
    )
)

rmsprop_mape = np.mean(
    np.abs(
        (rmsprop_actual.flatten() - rmsprop_pred.flatten())
        / rmsprop_actual.flatten()
    )
) * 100

rmsprop_r2 = r2_score(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_bias = np.mean(
    rmsprop_pred.flatten() -
    rmsprop_actual.flatten()
)

print("=== GRU + RMSPROP ===")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

=== GRU + RMSPROP ===
MAE : 1415.8972141398544
RMSE: 2049.566116899649
MAPE: 4.468899221948046
R²  : 0.8987310685819616
Bias: 240.38658371304984


**SGD**

In [31]:
from tensorflow.keras.optimizers import SGD

gru_sgd = Sequential([
    GRU(
        64,
        input_shape=(168, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

gru_sgd.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="mse",
    metrics=["mae"]
)

gru_sgd.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history_sgd = gru_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.5436 - mae: 0.5722 - val_loss: 0.3211 - val_mae: 0.4502
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.2492 - mae: 0.3942 - val_loss: 0.2016 - val_mae: 0.3527
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1801 - mae: 0.3322 - val_loss: 0.1711 - val_mae: 0.3204
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1562 - mae: 0.3060 - val_loss: 0.1555 - val_mae: 0.3028
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.1425 - mae: 0.2899 - val_loss: 0.1465 - val_mae: 0.2919
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.1332 - mae: 0.2785 - val_loss: 0.1394 - val_mae: 0.2838
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1265 - mae: 0.2699 - val_loss: 0.1331 - val_mae: 0.2755
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1213 - mae: 0.2632 - val_loss: 0.1286 - val_mae: 0.2704
Epoch 9/15
1586/1586 ━━━

In [33]:
sgd_pred_scaled = gru_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Prediction shape: (21593, 24)


In [34]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sgd_mae = mean_absolute_error(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_rmse = np.sqrt(
    mean_squared_error(
        sgd_actual.flatten(),
        sgd_pred.flatten()
    )
)

sgd_mape = np.mean(
    np.abs(
        (sgd_actual.flatten() - sgd_pred.flatten())
        / sgd_actual.flatten()
    )
) * 100

sgd_r2 = r2_score(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_bias = np.mean(
    sgd_pred.flatten() -
    sgd_actual.flatten()
)
print("=== GRU + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== GRU + SGD ===
MAE : 1688.3999903476658
RMSE: 2285.582024716131
MAPE: 5.413303277322066
R²  : 0.8740651336945433
Bias: 117.3761552706938


Learning Rate Scheduling

In [36]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

gru_lr = Sequential([
    GRU(
        64,
        input_shape=(168, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

gru_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [37]:
history_lr = gru_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.1187 - mae: 0.2490 - val_loss: 0.0941 - val_mae: 0.2243 - learning_rate: 0.0010
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0762 - mae: 0.1987 - val_loss: 0.0846 - val_mae: 0.2090 - learning_rate: 0.0010
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0695 - mae: 0.1878 - val_loss: 0.0865 - val_mae: 0.2124 - learning_rate: 0.0010
Epoch 4/15
1585/1586 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0664 - mae: 0.1826
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0655 - mae: 0.1815 - val_loss: 0.0933 - val_mae: 0.2259 - learning_rate: 0.0010
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0591 - mae: 0.1706 - val_loss: 0.0762 - val_mae: 0.1964 - learning_rate: 2.0000e-04
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0581 - mae: 0.1691 - val_loss: 0.0746 - val_mae: 0

In [38]:
lr_pred_scaled = gru_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Prediction shape: (21593, 24)


In [39]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [40]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr_mae = mean_absolute_error(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_rmse = np.sqrt(
    mean_squared_error(
        lr_actual.flatten(),
        lr_pred.flatten()
    )
)

lr_mape = np.mean(
    np.abs(
        (lr_actual.flatten() - lr_pred.flatten())
        / lr_actual.flatten()
    )
) * 100

lr_r2 = r2_score(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_bias = np.mean(
    lr_pred.flatten() -
    lr_actual.flatten()
)

print("==== GRU + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== GRU + RMSPROP + LearningRate ====
MAE : 1406.1677296106038
RMSE: 1965.4539756848421
MAPE: 4.466411028647664
R²  : 0.9068724625973502
Bias: 267.11585357764477


**Layers**

In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_deep = Sequential([

    GRU(
        64,
        return_sequences=True,
        input_shape=(168, 15)
    ),

    GRU(
        32,
        return_sequences=False
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

gru_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_6 (GRU)                     │ (None, 168, 64)        │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,632 (111.84 KB)

 Trainable params: 28,632 (111.84 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
history_deep = gru_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 31s 18ms/step - loss: 0.1214 - mae: 0.2468 - val_loss: 0.0883 - val_mae: 0.2179
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0701 - mae: 0.1904 - val_loss: 0.0808 - val_mae: 0.2060
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0642 - mae: 0.1807 - val_loss: 0.0808 - val_mae: 0.2053
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 41s 17ms/step - loss: 0.0608 - mae: 0.1752 - val_loss: 0.0801 - val_mae: 0.2031
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0584 - mae: 0.1712 - val_loss: 0.0733 - val_mae: 0.1934
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 41s 17ms/step - loss: 0.0560 - mae: 0.1675 - val_loss: 0.0760 - val_mae: 0.1966
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0537 - mae: 0.1643 - val_loss: 0.0814 - val_mae: 0.2007
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0512 - mae: 0.1607 - val_loss: 0.0777 - val_mae: 0.1958
Epoch 9/15
1586/1586 ━━━

In [43]:
deep_pred_scaled = gru_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
Prediction shape: (21593, 24)


In [44]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

deep_mae = mean_absolute_error(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_rmse = np.sqrt(
    mean_squared_error(
        deep_actual.flatten(),
        deep_pred.flatten()
    )
)

deep_mape = np.mean(
    np.abs(
        (deep_actual.flatten() - deep_pred.flatten())
        / deep_actual.flatten()
    )
) * 100

deep_r2 = r2_score(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_bias = np.mean(
    deep_pred.flatten() -
    deep_actual.flatten()
)

print("==== GRU + Layers ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

==== GRU + Layers ==========
MAE : 1530.9148826487824
RMSE: 2232.921070712413
MAPE: 4.813925651525387
R²  : 0.8798014840480364
Bias: 314.94480140229183


**Neurons**

In [46]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_128 = Sequential([

    GRU(
        128,
        input_shape=(168, 15)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

gru_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_128.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_8 (GRU)                     │ (None, 128)            │        55,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,496 (255.84 KB)

 Trainable params: 65,496 (255.84 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
history_128 = gru_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 0.1059 - mae: 0.2318 - val_loss: 0.0863 - val_mae: 0.2141
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0687 - mae: 0.1873 - val_loss: 0.0816 - val_mae: 0.2029
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0629 - mae: 0.1779 - val_loss: 0.0790 - val_mae: 0.1989
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0586 - mae: 0.1712 - val_loss: 0.0767 - val_mae: 0.1986
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0551 - mae: 0.1660 - val_loss: 0.0778 - val_mae: 0.1995
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0512 - mae: 0.1604 - val_loss: 0.0821 - val_mae: 0.2034
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0464 - mae: 0.1541 - val_loss: 0.0826 - val_mae: 0.2000
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0403 - mae: 0.1455 - val_loss: 0.0820 - val_mae: 0.1998
Epoch 9/15
1586/1586 ━━━

In [48]:
gru_128_pred_scaled = gru_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", gru_128_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21593, 24)


In [49]:
gru128_pred = target_scaler.inverse_transform(
    gru_128_pred_scaled.reshape(-1, 1)
).reshape(gru_128_pred_scaled.shape)

gru128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnn128_mae = mean_absolute_error(
    gru128_actual.flatten(),
    gru128_pred.flatten()
)

rnn128_rmse = np.sqrt(
    mean_squared_error(
        gru128_actual.flatten(),
        gru128_pred.flatten()
    )
)

rnn128_mape = np.mean(
    np.abs(
        (gru128_actual.flatten() - gru128_pred.flatten())
        / gru128_actual.flatten()
    )
) * 100

rnn128_r2 = r2_score(
    gru128_actual.flatten(),
    gru128_pred.flatten()
)

rnn128_bias = np.mean(
    gru128_pred.flatten() -
    gru128_actual.flatten()
)

print("=== GRU 128 NEURONS ===")
print("MAE :", rnn128_mae)
print("RMSE:", rnn128_rmse)
print("MAPE:", rnn128_mape)
print("R²  :", rnn128_r2)
print("Bias:", rnn128_bias)

=== GRU 128 NEURONS ===
MAE : 1543.5486209226153
RMSE: 2230.6602535482857
MAPE: 4.862702039128072
R²  : 0.8800447611621411
Bias: 305.5826138121416


32 **neurons**

In [51]:
gru_32 = Sequential([
    GRU(
        32,
        input_shape=(168, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

gru_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_32.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_9 (GRU)                     │ (None, 32)             │         4,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,376 (32.72 KB)

 Trainable params: 8,376 (32.72 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
history_32 = gru_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1422 - mae: 0.2696 - val_loss: 0.1011 - val_mae: 0.2317
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0778 - mae: 0.2022 - val_loss: 0.0885 - val_mae: 0.2140
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0710 - mae: 0.1906 - val_loss: 0.0838 - val_mae: 0.2084
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0675 - mae: 0.1848 - val_loss: 0.0817 - val_mae: 0.2048
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0652 - mae: 0.1812 - val_loss: 0.0786 - val_mae: 0.2024
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0634 - mae: 0.1782 - val_loss: 0.0826 - val_mae: 0.2018
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0617 - mae: 0.1753 - val_loss: 0.0759 - val_mae: 0.1957
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0607 - mae: 0.1738 - val_loss: 0.0770 - val_mae: 0.1949
Epoch 9/15
1586/1586 ━━━

In [53]:
gru32_pred_scaled = gru_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", gru32_pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
Prediction shape: (21593, 24)


In [54]:
gru32_pred = target_scaler.inverse_transform(
    gru32_pred_scaled.reshape(-1, 1)
).reshape(gru32_pred_scaled.shape)

gru32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [55]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

gru32_mae = mean_absolute_error(
    gru32_actual.flatten(),
    gru32_pred.flatten()
)

gru32_rmse = np.sqrt(
    mean_squared_error(
        gru32_actual.flatten(),
        gru32_pred.flatten()
    )
)

gru32_mape = np.mean(
    np.abs(
        (gru32_actual.flatten() - gru32_pred.flatten())
        / gru32_actual.flatten()
    )
) * 100

gru32_r2 = r2_score(
    gru32_actual.flatten(),
    gru32_pred.flatten()
)

gru32_bias = np.mean(
    gru32_pred.flatten() -
    gru32_actual.flatten()
)

print("=== GRU 32 NEURONS ===")
print("MAE :", gru32_mae)
print("RMSE:", gru32_rmse)
print("MAPE:", gru32_mape)
print("R²  :", gru32_r2)
print("Bias:", gru32_bias)

=== GRU 32 NEURONS ===
MAE : 1393.9605509769876
RMSE: 1965.0621137722849
MAPE: 4.4044384817786915
R²  : 0.9069095934561389
Bias: 133.02153184710227


Batch Size(32)

In [56]:
gru_batch32 = Sequential([
    GRU(64, input_shape=(168, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = gru_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step - loss: 0.1023 - mae: 0.2290 - val_loss: 0.0903 - val_mae: 0.2167
Epoch 2/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 33s 11ms/step - loss: 0.0695 - mae: 0.1882 - val_loss: 0.0818 - val_mae: 0.2055
Epoch 3/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0640 - mae: 0.1793 - val_loss: 0.0772 - val_mae: 0.1986
Epoch 4/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0608 - mae: 0.1744 - val_loss: 0.0760 - val_mae: 0.1973
Epoch 5/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.0581 - mae: 0.1700 - val_loss: 0.0761 - val_mae: 0.1934
Epoch 6/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0555 - mae: 0.1663 - val_loss: 0.0749 - val_mae: 0.1945
Epoch 7/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0532 - mae: 0.1632 - val_loss: 0.0750 - val_mae: 0.1952
Epoch 8/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0504 - mae: 0.1595 - val_loss: 0.0789 - val_mae: 0.2004
Epoch 9/15
3171/3171 ━━━

In [57]:
gru_batch32_pred_scaled = gru_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", gru_batch32_pred_scaled.shape)

675/675 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
Prediction shape: (21593, 24)


In [58]:
grubatch32_pred = target_scaler.inverse_transform(
    gru_batch32_pred_scaled.reshape(-1, 1)
).reshape(gru_batch32_pred_scaled.shape)

grubatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [59]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

grub32_mae = mean_absolute_error(
    grubatch32_actual.flatten(),
    grubatch32_pred.flatten()
)

grub32_rmse = np.sqrt(
    mean_squared_error(
        grubatch32_actual.flatten(),
        grubatch32_pred.flatten()
    )
)

grub32_mape = np.mean(
    np.abs(
        (grubatch32_actual.flatten() - grubatch32_pred.flatten())
        / grubatch32_actual.flatten()
    )
) * 100

grub32_r2 = r2_score(
    grubatch32_actual.flatten(),
    grubatch32_pred.flatten()
)

gru32_bias = np.mean(
    grubatch32_pred.flatten() -
    grubatch32_actual.flatten()
)

print("=== GRU batch32 NEURONS ===")
print("MAE :", grub32_mae)
print("RMSE:", grub32_rmse)
print("MAPE:", grub32_mape)
print("R²  :", grub32_r2)
print("Bias:", gru32_bias)

=== GRU batch32 NEURONS ===
MAE : 1561.4140228783574
RMSE: 2284.872485151488
MAPE: 4.894142464999258
R²  : 0.8741433123810879
Bias: 126.47026248544933


BatchSize(128)

In [60]:
gru_batch128 = Sequential([
    GRU(64, input_shape=(168, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = gru_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - loss: 0.1517 - mae: 0.2762 - val_loss: 0.0954 - val_mae: 0.2247
Epoch 2/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - loss: 0.0758 - mae: 0.1996 - val_loss: 0.0862 - val_mae: 0.2130
Epoch 3/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 0.0693 - mae: 0.1883 - val_loss: 0.0823 - val_mae: 0.2064
Epoch 4/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0657 - mae: 0.1824 - val_loss: 0.0797 - val_mae: 0.2029
Epoch 5/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - loss: 0.0633 - mae: 0.1785 - val_loss: 0.0806 - val_mae: 0.2005
Epoch 6/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0611 - mae: 0.1747 - val_loss: 0.0783 - val_mae: 0.1998
Epoch 7/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0593 - mae: 0.1718 - val_loss: 0.0766 - val_mae: 0.1954
Epoch 8/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - loss: 0.0579 - mae: 0.1698 - val_loss: 0.0767 - val_mae: 0.1960
Epoch 9/15
793/793 ━━━━━━━━━━━━━━━━━━━━ 9s 1

In [61]:
gru_batch128_pred_scaled = gru_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", gru_batch128_pred_scaled.shape)

169/169 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Prediction shape: (21593, 24)


In [62]:
batch128_pred = target_scaler.inverse_transform(
    gru_batch128_pred_scaled.reshape(-1, 1)
).reshape(gru_batch128_pred_scaled.shape)

batch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [63]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

grub128_mae = mean_absolute_error(
    batch128_actual.flatten(),
    batch128_pred.flatten()
)

grub128_rmse = np.sqrt(
    mean_squared_error(
        batch128_actual.flatten(),
        batch128_pred.flatten()
    )
)

grub128_mape = np.mean(
    np.abs(
        (batch128_actual.flatten() - batch128_pred.flatten())
        / batch128_actual.flatten()
    )
) * 100

grub128_r2 = r2_score(
    batch128_actual.flatten(),
    batch128_pred.flatten()
)

grub128_bias = np.mean(
    batch128_pred.flatten() -
    batch128_actual.flatten()
)

print("=== GRU batch128  ===")
print("MAE :", grub128_mae)
print("RMSE:", grub128_rmse)
print("MAPE:", grub128_mape)
print("R²  :", grub128_r2)
print("Bias:", grub128_bias)

=== GRU batch128  ===
MAE : 1484.7055919927934
RMSE: 2103.5083913969943
MAPE: 4.735002836469591
R²  : 0.8933303525953022
Bias: 447.11139531181857


**HyperParameter Tuning**

In [64]:
!pip install keras-tuner

In [65]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [66]:
def build_gru(hp):

    model = Sequential()

    model.add(
        GRU(
            units=hp.Choice(
                "rnn_units",
                values=[32, 64, 128]
            ),
            input_shape=(168, 15)
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [67]:
tuner = kt.RandomSearch(
    build_gru,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="gru_24_to_24"
)

Reloading Tuner from hyperparameter_tuning/gru_24_to_24/tuner0.json


In [68]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

In [69]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:",best_hp.get("rnn_units"))
print("Dense Units:",best_hp.get("dense_units"))
print("Learning Rate:",best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 64
Dense Units: 128
Learning Rate: 0.0005


In [70]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,968 (105.34 KB)

 Trainable params: 26,968 (105.34 KB)

 Non-trainable params: 0 (0.00 B)

In [71]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 33s 20ms/step - loss: 0.0530 - mae: 0.1611 - val_loss: 0.0714 - val_mae: 0.1856
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0521 - mae: 0.1597 - val_loss: 0.0733 - val_mae: 0.1902
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0511 - mae: 0.1582 - val_loss: 0.0774 - val_mae: 0.1957
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0503 - mae: 0.1572 - val_loss: 0.0747 - val_mae: 0.1912
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0494 - mae: 0.1560 - val_loss: 0.0764 - val_mae: 0.1951
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0482 - mae: 0.1545 - val_loss: 0.0737 - val_mae: 0.1920
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0471 - mae: 0.1530 - val_loss: 0.0729 - val_mae: 0.1882
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - loss: 0.0459 - mae: 0.1518 - val_loss: 0.0738 - val_mae: 0.1885
Epoch 9/15
1586/1586 ━━━

In [72]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step


In [73]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

best_mae = mean_absolute_error(
    best_actual.flatten(),
    best_pred.flatten()
)

best_rmse = np.sqrt(
    mean_squared_error(
        best_actual.flatten(),
        best_pred.flatten()
    )
)

best_mape = np.mean(
    np.abs(
        (best_actual.flatten() - best_pred.flatten())
        / best_actual.flatten()
    )
) * 100

best_r2 = r2_score(
    best_actual.flatten(),
    best_pred.flatten()
)

best_bias = np.mean(
    best_pred.flatten() -
    best_actual.flatten()
)

print("HYPERPARAMETER TUNED GRU")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED GRU
MAE : 1499.896177717129
RMSE: 2180.9395421997065
MAPE: 4.711323761127603
R²  : 0.8853326924493378
Bias: 289.6757637438143





**Additional Logs**

In [74]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [75]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144888, 21)


In [76]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [77]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101421, 21)
Validation: (21733, 21)
Test: (21734, 21)


In [78]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [79]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [80]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=168,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=168,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=168,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101230, 168, 21)
y_train: (101230, 24)
X_val: (21542, 168, 21)
y_val: (21542, 24)
X_test: (21543, 168, 21)
y_test: (21543, 24)


In [81]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_lags = Sequential([
    GRU(
        64,
        input_shape=(168, 21)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

gru_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_lags.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 64)             │        16,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,424 (87.59 KB)

 Trainable params: 22,424 (87.59 KB)

 Non-trainable params: 0 (0.00 B)

In [82]:
history_lags = gru_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.1147 - mae: 0.2411 - val_loss: 0.0865 - val_mae: 0.2147
Epoch 2/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0705 - mae: 0.1899 - val_loss: 0.0788 - val_mae: 0.2000
Epoch 3/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0644 - mae: 0.1798 - val_loss: 0.0770 - val_mae: 0.1968
Epoch 4/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0611 - mae: 0.1745 - val_loss: 0.0754 - val_mae: 0.1971
Epoch 5/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0585 - mae: 0.1703 - val_loss: 0.0742 - val_mae: 0.1923
Epoch 6/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0561 - mae: 0.1667 - val_loss: 0.0758 - val_mae: 0.1946
Epoch 7/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0541 - mae: 0.1638 - val_loss: 0.0788 - val_mae: 0.1949
Epoch 8/15
1582/1582 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0520 - mae: 0.1609 - val_loss: 0.0750 - val_mae: 0.1913
Epoch 9/15
1582/1582 ━━━

In [83]:
pred_scaled = gru_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

337/337 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Prediction shape: (21543, 24)


In [84]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [85]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(
    actual.flatten(),
    pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        actual.flatten(),
        pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (actual.flatten() - pred.flatten())
        / actual.flatten()
    )
) * 100

r2 = r2_score(
    actual.flatten(),
    pred.flatten()
)

bias = np.mean(
    pred.flatten() - actual.flatten()
)

print("=== GRU + ADDITIONAL LAGS ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== GRU + ADDITIONAL LAGS ===
MAE : 1514.6116841497178
RMSE: 2196.4882903684506
MAPE: 4.770651087221549
R²  : 0.8836414306178983
Bias: 295.5581584945117


In [1]:
import pandas as pd
import os

gru_168_24_results = [
    ["GRU Baseline", 168, 24, 1452.6757279173771, 2075.132045638438, 4.598523493680257, 0.8961888896859629, 284.29167971810284],
    ["GRU + EarlyStopping", 168, 24, 1409.7967050825873, 1980.18629570404, 4.480879707174862, 0.9054711307453631, 244.07954247340598],
    ["GRU + Dropout", 168, 24, 1483.847755260765, 2052.339182603539, 4.750156463961595, 0.89845684919783, 293.26455467293727],
    ["GRU + Batch Normalization", 168, 24, 1491.3439278771682, 2121.1136738033306, 4.7553097408570695, 0.8915373404934451, 420.75683135084165],
    ["GRU + RMSprop", 168, 24, 1415.8972141398544, 2049.566116899649, 4.468899221948046, 0.8987310685819616, 240.38658371304984],
    ["GRU + SGD", 168, 24, 1688.3999903476658, 2285.582024716131, 5.413303277322066, 0.8740651336945433, 117.3761552706938],
    ["GRU + RMSprop + Learning Rate", 168, 24, 1406.1677296106038, 1965.4539756848421, 4.466411028647664, 0.9068724625973502, 267.11585357764477],
    ["GRU + Additional Layers", 168, 24, 1530.9148826487824, 2232.921070712413, 4.813925651525387, 0.8798014840480364, 314.94480140229183],
    ["GRU 128 Neurons", 168, 24, 1543.5486209226153, 2230.6602535482857, 4.862702039128072, 0.8800447611621411, 305.5826138121416],
    ["GRU 32 Neurons", 168, 24, 1393.9605509769876, 1965.0621137722849, 4.4044384817786915, 0.9069095934561389, 133.02153184710227],
    ["GRU Batch Size 32", 168, 24, 1561.4140228783574, 2284.872485151488, 4.894142464999258, 0.8741433123810879, 126.47026248544933],
    ["GRU Batch Size 128", 168, 24, 1484.7055919927934, 2103.5083913969943, 4.735002836469591, 0.8933303525953022, 447.11139531181857],
    ["GRU Hyperparameter Tuned", 168, 24, 1499.896177717129, 2180.9395421997065, 4.711323761127603, 0.8853326924493378, 289.6757637438143],
    ["GRU + Additional Lags", 168, 24, 1514.6116841497178, 2196.4882903684506, 4.770651087221549, 0.8836414306178983, 295.5581584945117]
]

gru_168_24_df = pd.DataFrame(
    gru_168_24_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

# Display with 6 decimal places
display(
    gru_168_24_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

# Save inside existing comparison folder
os.makedirs("comparison", exist_ok=True)

gru_168_24_df.to_csv(
    "comparison/GRU_168h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved: comparison/GRU_168h_to_24h_results.csv")

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,GRU Baseline,168,24,1452.675728,2075.132046,4.598523,0.896189,284.291680
1,GRU + EarlyStopping,168,24,1409.796705,1980.186296,4.480880,0.905471,244.079542
2,GRU + Dropout,168,24,1483.847755,2052.339183,4.750156,0.898457,293.264555
3,GRU + Batch Normalization,168,24,1491.343928,2121.113674,4.755310,0.891537,420.756831
4,GRU + RMSprop,168,24,1415.897214,2049.566117,4.468899,0.898731,240.386584
5,GRU + SGD,168,24,1688.399990,2285.582025,5.413303,0.874065,117.376155
6,GRU + RMSprop + Learning Rate,168,24,1406.167730,1965.453976,4.466411,0.906872,267.115854
7,GRU + Additional Layers,168,24,1530.914883,2232.921071,4.813926,0.879801,314.944801
8,GRU 128 Neurons,168,24,1543.548621,2230.660254,4.862702,0.880045,305.582614
9,GRU 32 Neurons,168,24,1393.960551,1965.062114,4.404438,0.906910,133.021532


Saved: comparison/GRU_168h_to_24h_results.csv


In [2]:
best_gru_168_24 = gru_168_24_df.loc[gru_168_24_df["Model"] == "GRU 32 Neurons"]
display(best_gru_168_24.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "MAPE (%)": "{:.6f}",
        "R²": "{:.6f}",
        "Bias": "{:.6f}"
    })
)

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
9,GRU 32 Neurons,168,24,1393.960551,1965.062114,4.404438,0.906910,133.021532
